In [1]:
import os
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_mean_pool
from torch_geometric.loader import DataLoader
from tqdm import tqdm
import warnings
from datetime import date
warnings.filterwarnings('ignore')

In [8]:
# --- CONFIGURATION ---
INPUT_DIR = "/home/nba9/data_competition/processed_data/"
MODEL_DIR = "/home/nba9/data_competition/saved_models_STAR/"
SUBMISSION_FILE = MODEL_DIR +"submission.csv"
WETLAB_FILE = MODEL_DIR+"wetlab_recommendations.csv"
# load config.txt if it exists, otherwise use defaults
config = {
    "ENSEMBLE_SIZE": 5,
    "HIDDEN_DIM": 256,
    "ESM_DIM": 1280,
    "NODE_FEAT": 29,
    "EDGE_FEAT": 7,
     "N_GIN_LAYERS": 4,
     "DROPOUT": 0.2,
     "BATCH_SIZE": 128}
if os.path.exists(MODEL_DIR + "config.txt"):
    with open(MODEL_DIR + "config.txt", "r") as f:
        for line in f:
            key, value = line.strip().split("=")
            config[key] = int(value)

MC_SAMPLES_PER_MODEL = 10 # 5 models * 10 samples = 50 total stochastic passes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
class DTA_Model(nn.Module):
    """
    Drug-Target Affinity predictor modified for Uncertainty Estimation.
    """

    def __init__(self, node_feat=29, edge_feat=7, esm_dim=1280,
                 hidden_dim=256, n_layers=4, dropout=0.2):
        super().__init__()
        self.dropout = dropout

        # --- Drug graph encoder (GINEConv) ---
        self.edge_proj = nn.Linear(edge_feat, hidden_dim)

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for i in range(n_layers):
            in_dim = node_feat if i == 0 else hidden_dim
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(mlp, edge_dim=hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        # --- Drug fingerprint encoder ---
        self.fp_proj = nn.Sequential(
            nn.Linear(2048, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # --- Protein encoder ---
        self.prot_proj = nn.Sequential(
            nn.Linear(esm_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # --- Prediction head ---
        # graph(256) + fp(256) + protein(256) = 768
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            # MODIFICATION: Output 2 values (mean, log_variance)
            nn.Linear(hidden_dim // 2, 2),
        )

    def get_compound_embedding(self, data):
        """Extract the compound graph embedding (used by Phase 4 for OOD detection)."""
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        edge_emb = self.edge_proj(edge_attr)
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_emb)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return global_mean_pool(x, batch)

    def forward(self, data):
        # 1. Drug graph
        drug_graph = self.get_compound_embedding(data)  # (B, hidden)
        # 2. Drug fingerprint
        drug_fp = self.fp_proj(data.fp.squeeze(1))      # (B, hidden)
        # 3. Protein
        prot = self.prot_proj(data.target_emb.squeeze(1))  # (B, hidden)
        # 4. Predict
        combined = torch.cat([drug_graph, drug_fp, prot], dim=-1)
        out = self.head(combined)

        # Split output into mean affinity and aleatoric uncertainty
        mu = out[:, 0]
        logvar = out[:, 1]

        return mu, logvar

In [13]:
# --- STAR HELPER FUNCTIONS ---
def ensemble_mc_inference(models, data_loader, device, mc_samples=10, has_targets=False, save_path=None):
    """S: Computes Expected Affinity and Total Variance via Deep Ensembles + MC Dropout."""
    # Force all models into TRAIN mode so Dropout remains active for MC sampling
    for model in models: model.train()
    
    all_results = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Running Stochastic Inference"):
            batch = batch.to(device)
            batch_mus, batch_logvars = [], []
            
            # Pass through every model in the ensemble `mc_samples` times
            for model in models:
                for _ in range(mc_samples):
                    mu, logvar = model(batch)
                    batch_mus.append(mu)
                    batch_logvars.append(logvar)
            
            # Stack to dimensions: [Total_Passes, Batch_Size]
            batch_mus = torch.stack(batch_mus)
            batch_logvars = torch.stack(batch_logvars)
            
            # Calculate Expectations and Variances
            expected_mu = torch.mean(batch_mus, dim=0)
            epistemic_var = torch.var(batch_mus, dim=0)
            aleatoric_var = torch.mean(torch.exp(batch_logvars), dim=0)
            total_var = epistemic_var + aleatoric_var
            
            # Extract IDs and Split Flags safely
            batch_ids = batch.sample_id.cpu().numpy() if torch.is_tensor(batch.sample_id) else batch.sample_id
            
            for i in range(len(batch_ids)):
                row = {
                    'id': batch_ids[i],
                    'expected_affinity': expected_mu[i].item(),
                    'total_variance': total_var[i].item(),
                    'std_dev': np.sqrt(total_var[i].item()),
                    'in_A': batch.in_A[i].item() if hasattr(batch, 'in_A') else 0,
                    'in_B': batch.in_B[i].item() if hasattr(batch, 'in_B') else 0,
                    'in_C': batch.in_C[i].item() if hasattr(batch, 'in_C') else 0,
                }
                if has_targets: row['true_affinity'] = batch.y[i].item()
                all_results.append(row)
            
            if save_path:
                pd.DataFrame(all_results).to_csv(save_path, index=False)
                
    return pd.DataFrame(all_results)

def report_degradation(val_df, save_path=None):
    """T & R: Evaluates and reports Shift-Aware performance using the Validation Set."""
    print("\n" + "="*50)
    print("STAR REPORT: SHIFT-AWARE DEGRADATION (T & R)")
    print("="*50)
    
    def calc_metrics(df_subset, name):
        if len(df_subset) == 0: return None
        mse = np.mean((df_subset['expected_affinity'] - df_subset['true_affinity'])**2)
        return {'Split': name, 'N': len(df_subset), 'RMSE': np.sqrt(mse), 'Avg_Uncertainty': df_subset['std_dev'].mean()}
    
    stats = []
    stats.append(calc_metrics(val_df[val_df['in_A'] == 1], "Split A (IID/Base)"))
    stats.append(calc_metrics(val_df[val_df['in_B'] == 1], "Split B (Scaffold Shift)"))
    stats.append(calc_metrics(val_df[val_df['in_C'] == 1], "Split C (Assay Shift)"))
    
    stats_df = pd.DataFrame([s for s in stats if s])
    print(stats_df.to_string(index=False))
    
    if len(stats_df) > 1:
        base_rmse = stats_df[stats_df['Split'].str.contains('A')]['RMSE'].values[0]
        shift_rmse = stats_df[stats_df['Split'].str.contains('B')]['RMSE'].values[0]
        deg = ((shift_rmse - base_rmse) / base_rmse) * 100
        print(f"\n Degradation Alert: Performance drops by {deg:.1f}% when encountering novel scaffolds (Split B).")
        print("This highlights the expected safety margin when screening entirely new chemical spaces.")
    print("="*50 + "\n")
    
    if save_path:
        stats_df.to_csv(save_path, index=False)

def select_candidates_for_wetlab(test_df, budget=100, lambda_risk=1.5, max_variance=3.0):
    """A: Ranks blind candidates by Expected Utility and enforces Abstention."""
    print(f"Applying Wet-Lab Budget constraints (Budget: {budget}, Risk Lambda: {lambda_risk})...")
    
    # Abstention Policy (Reject hallucinations, i.e. compounds with very high uncertainty)
    eligible_df = test_df[test_df['total_variance'] <= max_variance].copy()
    rejected = len(test_df) - len(eligible_df)
    print(f"-> Abstention Policy triggered: Dropped {rejected} compoundd that are two uncertain.")
    
    # Calculate Utility: U(x) = predicted_affinity - (lambda * standard_deviation)
    eligible_df['utility'] = eligible_df['expected_affinity'] - (lambda_risk * eligible_df['std_dev'])
    
    # Sort by utility and select
    ranked_df = eligible_df.sort_values(by='utility', ascending=False)
    final_picks = ranked_df.head(budget)
    
    print(f"-> Selected top {len(final_picks)} compounds for synthesis/assay.")
    return final_picks[['id', 'expected_affinity', 'std_dev', 'utility']]

from scipy.stats import kendalltau
import numpy as np

def run_star_evaluation_metrics(val_df, output_file = 'star_evaluation_report.txt'):
    """
    Executes the 4 core STAR framework metrics using the pre-computed validation DataFrame.
    """
    report_lines = []
    def log(text=""):
        print(text)
        report_lines.append(text)
        
    print('\n' + '='*60)
    print('STAR FRAMEWORK: EVALUATION METRICS')
    print('='*60)
    
    # Extract arrays directly from our Phase 4 inference DataFrame
    val_preds = val_df['expected_affinity'].values
    val_targets = val_df['true_affinity'].values
    val_std = val_df['std_dev'].values
    
    # We use our combined epistemic + aleatoric variance as the OOD signal
    val_ood = val_df['total_variance'].values 
    
    split_flags = {
        'A': val_df['in_A'].values == 1, 
        'B': val_df['in_B'].values == 1, 
        'C': val_df['in_C'].values == 1
    }

    print(f'Val samples: {len(val_targets)} | RMSE: {np.sqrt(np.mean((val_preds - val_targets)**2)):.4f}\n')

    # ─────────────────────────────────────────────────────────────────────
    # METRIC 1: Ranking Stability (Kendall's tau)
    # ─────────────────────────────────────────────────────────────────────
    print("=== 1. Ranking Stability (Kendall's tau, top-200) ===")
    K = min(200, len(val_targets) // 4)
    tau_vals = []
    
    for split, mask in split_flags.items():
        if mask.sum() == 0: continue
        sp, st = val_preds[mask], val_targets[mask]
        
        if mask.sum() < K:
            topk = np.arange(len(sp))
        else:
            topk = np.argsort(sp)[-K:] 
            
        tau, pval = kendalltau(sp[topk], st[topk])
        tau_vals.append(tau)
        print(f'  Split {split} (n={mask.sum()}):  tau = {tau:.4f}  (p = {pval:.3g})')

    # ─────────────────────────────────────────────────────────────────────
    # METRIC 2: Hit Rate @ Budget
    # ─────────────────────────────────────────────────────────────────────
    print('\n=== 2. Hit Rate @ Budget (top-200 predictions) ===')
    HIT_THRESHOLD = np.percentile(val_targets, 75)
    BUDGET = min(200, len(val_targets) // 4)
    print(f'  Hit threshold (75th pct of val affinity): {HIT_THRESHOLD:.2f}')
    
    for split, mask in split_flags.items():
        if mask.sum() == 0: continue
        sp, st = val_preds[mask], val_targets[mask]
        
        true_hits  = st >= HIT_THRESHOLD
        budget     = min(BUDGET, len(sp))
        topk       = np.argsort(sp)[-budget:]
        hits_found = true_hits[topk].sum()
        total_hits = true_hits.sum()
        
        hit_rate   = hits_found / budget
        recall     = hits_found / total_hits if total_hits > 0 else 0.0
        print(f'  Split {split}: Hit Rate@{budget} = {hit_rate:.3f}  |  Recall = {recall:.3f}  ({hits_found}/{total_hits} hits)')

    # ─────────────────────────────────────────────────────────────────────
    # METRIC 3: Calibration (ECE via Gaussian interval coverage)
    # ─────────────────────────────────────────────────────────────────────
    print('\n=== 3. Calibration (MC Dropout interval coverage) ===')
    errors = val_targets - val_preds 
    w1 = (np.abs(errors) < 1.0 * val_std).mean()
    w2 = (np.abs(errors) < 2.0 * val_std).mean()
    w3 = (np.abs(errors) < 3.0 * val_std).mean()
    
    ece = (abs(w1 - 0.683) + abs(w2 - 0.954) + abs(w3 - 0.997)) / 3
    print(f'  Within 1 sigma: {w1:.3f}  (ideal 0.683)')
    print(f'  Within 2 sigma: {w2:.3f}  (ideal 0.954)')
    print(f'  Within 3 sigma: {w3:.3f}  (ideal 0.997)')
    print(f'  ECE (mean |observed - expected|): {ece:.4f}')
    
    if w1 > 0.683:
        print('  -> Model is under-confident (uncertainty intervals are wider than necessary)')
    else:
        print('  -> Model is over-confident (uncertainty intervals are too tight)')

    # ─────────────────────────────────────────────────────────────────────
    # METRIC 4: Abstention Quality
    # ─────────────────────────────────────────────────────────────────────
    print('\n=== 4. Abstention Quality (Variance flag -> high prediction error) ===')
    val_abs_err   = np.abs(errors)
    err_threshold = np.percentile(val_abs_err, 75)
    ood_threshold = np.percentile(val_ood, 75) 

    high_ood   = val_ood > ood_threshold
    high_error = val_abs_err > err_threshold

    TP = (high_ood & high_error).sum()
    FP = (high_ood & ~high_error).sum()
    FN = (~high_ood & high_error).sum()

    precision  = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall_abs = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2*precision*recall_abs / (precision+recall_abs) if (precision+recall_abs) > 0 else 0.0

    print(f'  Error threshold (75th pct):  {err_threshold:.3f} pKi units')
    print(f'  Variance threshold (75th pct): {ood_threshold:.3f}')
    print(f'  Precision: {precision:.3f}  |  Recall: {recall_abs:.3f}  |  F1: {f1:.3f}')
    print(f'  (Random-flag baseline precision would be 0.250)')

    # ─────────────────────────────────────────────────────────────────────
    # SUMMARY TABLE
    # ─────────────────────────────────────────────────────────────────────
    print('\n' + '='*60)
    print('PIPELINE EVALUATION SUMMARY (Against Success Criteria)')
    print('='*60)
    print(f'  Ranking Stability (mean tau): {np.mean(tau_vals):.4f} (Target: ≥ 0.6)')
    print(f'  Calibration ECE:              {ece:.4f} (Target: < 0.10)')
    print(f'  Abstention Precision:         {precision:.3f} (Target: ≥ 0.80)')
    print('='*60 + '\n')

    try:
        with open(output_file, 'w') as f:
            f.write("\n".join(report_lines))
        print(f"STAR Evaluation Report successfully saved to: {output_file}")
    except Exception as e:
        print(f"Failed to save report to {output_file}. Error: {e}")

In [6]:
# --- EXECUTION ---
print("Loading datasets and models...")
val_data = torch.load(os.path.join(INPUT_DIR, "val_data.pt"))
test_data = torch.load(os.path.join(INPUT_DIR, "test_data.pt"))

val_loader = DataLoader(val_data, batch_size=config["BATCH_SIZE"], shuffle=False)
test_loader = DataLoader(test_data, batch_size=config["BATCH_SIZE"], shuffle=False)

models = []
for i in range(1, config["ENSEMBLE_SIZE"] + 1):
    m = DTA_Model(node_feat=config["NODE_FEAT"], 
                  edge_feat=config["EDGE_FEAT"],
                  esm_dim=config["ESM_DIM"],
                  hidden_dim=config["HIDDEN_DIM"],
                  n_layers=config["N_GIN_LAYERS"],
                  dropout=config["DROPOUT"]).to(device)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'model_{i}.pt'), map_location=device))
    models.append(m)

Loading datasets and models...


In [9]:
# Validation Set Analysis (T & R)
print("\n--- Phase 4, Step 1: Internal Validation Analysis ---")
val_results_df = ensemble_mc_inference(models, val_loader, device, mc_samples=MC_SAMPLES_PER_MODEL, has_targets=True, save_path=MODEL_DIR+"val_predictions.csv")
report_degradation(val_results_df, save_path=MODEL_DIR+"val_degradation_report.csv")


--- Phase 4, Step 1: Internal Validation Analysis ---


Running Stochastic Inference: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:16<00:00,  1.25it/s]


STAR REPORT: SHIFT-AWARE DEGRADATION (T & R)
                   Split    N     RMSE  Avg_Uncertainty
      Split A (IID/Base) 1804 0.558697         0.567327
Split B (Scaffold Shift) 1782 0.592485         0.614165
   Split C (Assay Shift) 2578 0.558007         0.568717

 Degradation Alert: Performance drops by 0.0% when encountering novel scaffolds (Split B).
This highlights the expected safety margin when screening entirely new chemical spaces.



In [14]:
run_star_evaluation_metrics(val_results_df, output_file=MODEL_DIR + "star_evaluation_report.txt")


STAR FRAMEWORK: EVALUATION METRICS
Val samples: 2578 | RMSE: 0.5580

=== 1. Ranking Stability (Kendall's tau, top-200) ===
  Split A (n=1804):  tau = 0.2592  (p = 5.5e-08)
  Split B (n=1782):  tau = 0.2633  (p = 3.43e-08)
  Split C (n=2578):  tau = 0.2751  (p = 7.96e-09)

=== 2. Hit Rate @ Budget (top-200 predictions) ===
  Hit threshold (75th pct of val affinity): 5.44
  Split A: Hit Rate@200 = 0.910  |  Recall = 0.396  (182/460 hits)
  Split B: Hit Rate@200 = 0.920  |  Recall = 0.352  (184/523 hits)
  Split C: Hit Rate@200 = 0.930  |  Recall = 0.287  (186/649 hits)

=== 3. Calibration (MC Dropout interval coverage) ===
  Within 1 sigma: 0.910  (ideal 0.683)
  Within 2 sigma: 0.978  (ideal 0.954)
  Within 3 sigma: 0.995  (ideal 0.997)
  ECE (mean |observed - expected|): 0.0843
  -> Model is under-confident (uncertainty intervals are wider than necessary)

=== 4. Abstention Quality (Variance flag -> high prediction error) ===
  Error threshold (75th pct):  0.454 pKi units
  Variance t

In [10]:
print("\n--- Phase 4, Step 2: Blind Test Set Inference ---")
test_results_df = ensemble_mc_inference(models, test_loader, device, mc_samples=MC_SAMPLES_PER_MODEL, has_targets=False)


--- Phase 4, Step 2: Blind Test Set Inference ---


Running Stochastic Inference: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 182/182 [02:21<00:00,  1.29it/s]


In [15]:
# Kaggle Submission
submission_df = test_results_df[['id', 'expected_affinity']].rename(columns={'expected_affinity': 'affinity'})
submission_df.to_csv(SUBMISSION_FILE, index=False)
print(f"\n Submission saved to: {SUBMISSION_FILE}")


 Submission saved to: /home/nba9/data_competition/saved_models_STAR/submission.csv


In [16]:
# Wet-Lab Recommendations (A)
wetlab_df = select_candidates_for_wetlab(test_results_df, budget=100, lambda_risk=1.5)
wetlab_df.to_csv(WETLAB_FILE, index=False)
print(f" Wet-Lab recommended list saved to: {WETLAB_FILE}")
print("\nTop 5 Wet-Lab Recommendations:")
print(wetlab_df.head())

Applying Wet-Lab Budget constraints (Budget: 100, Risk Lambda: 1.5)...
-> Abstention Policy triggered: Dropped 171 compoundd that are two uncertain.
-> Selected top 100 compounds for synthesis/assay.
 Wet-Lab recommended list saved to: /home/nba9/data_competition/saved_models_STAR/wetlab_recommendations.csv

Top 5 Wet-Lab Recommendations:
               id  expected_affinity   std_dev   utility
836    D_23530495           8.843350  1.506796  6.583156
14547  D_62971438           8.487970  1.376837  6.422715
16116  D_46249845           8.626775  1.568728  6.273683
20299  D_94728082           8.827473  1.709968  6.262521
7084   D_34635056           8.311071  1.381579  6.238703
